# BFA Attack Visualization

This notebook visualizes the results of the Bit-Flip Attack on ResNet-20.

In [ ]:
# Imports
import sys
sys.path.append('..')

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
from bfa.evaluate_attack import (
    load_attack_log,
    get_attack_metrics,
    generate_attack_report,
    plot_attack_history,
    plot_bit_flip_heatmap
)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

## 1. Load Attack Log

In [ ]:
# Path to attack log
log_path = '../results/attack_log.pkl'

# Check if file exists
if not os.path.exists(log_path):
    print(f"Error: Attack log not found at {log_path}")
    print("Please run the attack first: bash scripts/run_attack.sh")
else:
    log = load_attack_log(log_path)
    history = log['history']
    flip_summary = log.get('flip_summary', {})
    
    print("Attack log loaded successfully!")
    print(f"Rounds: {len(history['rounds']) - 1}")
    print(f"Total flips: {history['flips'][-1]}")

## 2. Attack Summary

In [ ]:
# Display key metrics
metrics = get_attack_metrics(log_path)

print("="*50)
print("BFA Attack Summary")
print("="*50)
print(f"Initial accuracy: {metrics['initial_accuracy']:.2f}%")
print(f"Final accuracy:   {metrics['final_accuracy']:.2f}%")
print(f"Accuracy drop:    {metrics['accuracy_drop']:.2f}%")
print(f"Initial loss:     {metrics['initial_loss']:.4f}")
print(f"Final loss:       {metrics['final_loss']:.4f}")
print(f"Total bit flips:  {metrics['total_flips']}")
print(f"Total rounds:     {metrics['total_rounds']}")
print("="*50)

## 3. Accuracy vs Bit Flips

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(history['flips'], history['accuracy'], 'b-', linewidth=2, label='Accuracy')
ax.axhline(y=10, color='r', linestyle='--', label='Target (10%)')
ax.set_xlabel('Cumulative Bit Flips')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy vs Bit Flips')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 4. Loss vs Bit Flips

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(history['flips'], history['loss'], 'r-', linewidth=2)
ax.set_xlabel('Cumulative Bit Flips')
ax.set_ylabel('Loss')
ax.set_title('Model Loss vs Bit Flips')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Combined Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history['flips'], history['accuracy'], 'b-', linewidth=2)
ax1.set_xlabel('Cumulative Bit Flips')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Model Accuracy vs Bit Flips')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=10, color='r', linestyle='--', label='Target (10%)')
ax1.legend()

# Loss plot
ax2.plot(history['flips'], history['loss'], 'r-', linewidth=2)
ax2.set_xlabel('Cumulative Bit Flips')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss vs Bit Flips')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Bit Flip Heatmap (by Layer and Position)

In [ ]:
if flip_summary:
    # Get unique layers
    layers = list(flip_summary.keys())
    bit_positions = list(range(32))
    
    # Create data matrix
    data = np.zeros((len(layers), 32))
    for i, layer in enumerate(layers):
        for bit_pos in bit_positions:
            data[i, bit_pos] = flip_summary[layer].get(bit_pos, 0)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, max(6, len(layers) * 0.5)))
    
    im = ax.imshow(data, aspect='auto', cmap='YlOrRd')
    
    # Set ticks
    ax.set_xticks(bit_positions)
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels([l.replace('module.', '') for l in layers])
    
    # Add bit region labels
    ax.axvline(x=-0.5, color='white', linewidth=2)
    ax.axvline(x=22.5, color='white', linewidth=2)
    ax.axvline(x=30.5, color='white', linewidth=2)
    
    ax.text(11, -1, 'Mantissa', ha='center', va='bottom')
    ax.text(26.5, -1, 'Exponent', ha='center', va='bottom')
    ax.text(31, -1, 'S', ha='center', va='bottom')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Number of Flips')
    
    ax.set_xlabel('Bit Position')
    ax.set_ylabel('Layer')
    ax.set_title('Bit Flip Distribution by Layer and Position')
    
    plt.tight_layout()
    plt.show()
else:
    print("No flip summary data available.")

## 7. Bit Flip Distribution by Bit Type

In [ ]:
if flip_summary:
    # Aggregate flips by bit type
    sign_flips = 0
    exp_flips = 0
    mant_flips = 0
    
    for layer_name, bit_counts in flip_summary.items():
        sign_flips += bit_counts.get(31, 0)
        exp_flips += sum(bit_counts.get(b, 0) for b in range(23, 31))
        mant_flips += sum(bit_counts.get(b, 0) for b in range(23))
    
    total = sign_flips + exp_flips + mant_flips
    
    # Create pie chart
    fig, ax = plt.subplots(figsize=(8, 8))
    
    labels = [f'Sign bit\n({sign_flips})',
              f'Exponent bits\n({exp_flips})',
              f'Mantissa bits\n({mant_flips})']
    sizes = [sign_flips, exp_flips, mant_flips]
    colors = ['#ff6b6b', '#feca57', '#48dbfb']
    explode = (0.05, 0.05, 0.05)
    
    ax.pie(sizes, explode=explode, labels=labels, colors=colors,
           autopct='%1.1f%%', shadow=True, startangle=90)
    ax.set_title(f'Bit Flip Distribution (Total: {total} flips)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No flip summary data available.")

## 8. Per-Round Analysis

In [ ]:
# Create per-round statistics
rounds = np.array(history['rounds'][1:])  # Skip initial state
flips = np.array(history['flips'][1:])
accuracy = np.array(history['accuracy'][1:])
loss = np.array(history['loss'][1:])

# Compute per-round changes
flips_per_round = np.diff(flips, prepend=0)
acc_drop_per_round = np.diff(accuracy, prepend=accuracy[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Flips per round
ax1.bar(rounds, flips_per_round, color='steelblue', alpha=0.7)
ax1.set_xlabel('Round')
ax1.set_ylabel('Flips per Round')
ax1.set_title('Bit Flips per Round')
ax1.grid(True, alpha=0.3, axis='y')

# Accuracy drop per round
ax2.bar(rounds, -acc_drop_per_round, color='coral', alpha=0.7)
ax2.set_xlabel('Round')
ax2.set_ylabel('Accuracy Drop per Round (%)')
ax2.set_title('Accuracy Drop per Round')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Save Plots

In [ ]:
# Create output directory
os.makedirs('../results/plots', exist_ok=True)

# Save all plots
plot_attack_history(history, save_path='../results/plots/attack_history.png')

if flip_summary:
    plot_bit_flip_heatmap(flip_summary, save_path='../results/plots/bit_flip_heatmap.png')

print("Plots saved to ../results/plots/")

## 10. Generate Text Report

In [ ]:
# Generate and display report
report = generate_attack_report(
    history, 
    flip_summary=flip_summary if flip_summary else None,
    save_path='../results/attack_report.txt'
)

print(report)